# ⚙️ Day 4B — Extraction Engine: Function Calling on Real Documents
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Where We Are

Notebook 4A gave us a robust `TaxDocument` Pydantic schema with:
- Optional fields for everything a document might omit
- `missing_fields` tracker
- `extraction_confidence` (not answer confidence)
- Auto review-flagging on critical missing fields

**This notebook:** we plug that schema into Azure OpenAI's **function calling** API.  
Function calling enforces the JSON schema at the model layer — the model cannot deviate from it.
No fences to strip. No prose to parse around.

We test on 5 documents of increasing difficulty:
1. Clean, well-structured email
2. Ambiguous letter with approximate figures
3. Invoice snippet with missing period
4. Multi-entity document (two companies, two transactions)
5. Near-empty message (extreme partial extraction)

---
## ⏱️ Time: 60 minutes

---
## ⚙️ Step 1: Install & Import

In [ ]:
%pip install openai pydantic --quiet

In [ ]:
import os
import re
import json
import datetime
from typing import Literal, Optional
from openai import AzureOpenAI
from getpass import getpass
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI Client

In [ ]:
AZURE_OPENAI_ENDPOINT = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY  = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME = input("Deployment name (e.g. gpt-4o): ").strip()
AZURE_API_VERSION     = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)

print("✅ Azure OpenAI client initialised successfully!")

---
## 📦 Step 3: Load Schema from Notebook 4A

If you haven't run 4A yet, the TaxDocument class and schema are also defined inline below.

In [ ]:
# ── TaxDocument Pydantic model (full definition — self-contained) ──────────────

class TaxDocument(BaseModel):
    entity_name:           Optional[str]   = Field(default=None)
    tax_period:            Optional[str]   = Field(default=None)
    transaction_type:      Optional[str]   = Field(default=None)
    jurisdiction:          Optional[str]   = Field(default=None)
    liability_amount:      Optional[float] = Field(default=None)
    currency: Literal["INR","USD","EUR","GBP","AED","SGD","Unknown"] = Field(default="INR")
    applicable_section:    Optional[str]   = Field(default=None)
    extraction_confidence: float           = Field(ge=0.0, le=1.0)
    missing_fields:        list[str]       = Field(default_factory=list)
    extraction_notes:      Optional[str]   = Field(default=None)
    human_review_required: bool            = Field(default=False)

    @field_validator("currency", mode="before")
    @classmethod
    def normalise_currency(cls, v):
        if v is None: return "INR"
        mapping = {
            "rupee":"INR","rupees":"INR","rs":"INR","inr":"INR","₹":"INR",
            "dollar":"USD","dollars":"USD","usd":"USD","$":"USD",
            "euro":"EUR","eur":"EUR","€":"EUR",
            "pound":"GBP","gbp":"GBP","£":"GBP",
            "dirham":"AED","aed":"AED","sgd":"SGD",
        }
        return mapping.get(str(v).lower().strip(), str(v).upper() if str(v).upper() in ["INR","USD","EUR","GBP","AED","SGD"] else "Unknown")

    @field_validator("liability_amount", mode="before")
    @classmethod
    def parse_amount(cls, v):
        if v is None: return None
        if isinstance(v, (int, float)): return float(v)
        s = str(v).lower().replace(",", "").replace("₹", "").replace("rs", "").strip()
        m = re.search(r"([\d.]+)\s*l(?:akh)?", s)
        if m: return float(m.group(1)) * 100000
        m = re.search(r"([\d.]+)\s*cr(?:ore)?", s)
        if m: return float(m.group(1)) * 10000000
        m = re.search(r"[\d.]+", s)
        return float(m.group()) if m else None

    @model_validator(mode="after")
    def auto_set_review_flag(self):
        critical = {"entity_name", "tax_period", "transaction_type"}
        if self.extraction_confidence < 0.75 or critical.intersection(set(self.missing_fields)):
            self.human_review_required = True
        return self


# ── Load function schema ───────────────────────────────────────────────────────
try:
    with open("day4_schema.json") as f:
        EXTRACTION_FUNCTION_SCHEMA = json.load(f)
    print("✅ Loaded function schema from day4_schema.json")
except FileNotFoundError:
    print("⚠️  day4_schema.json not found — run Notebook 4A first, or use inline definition below")

---
## 🔧 Step 4: Build the Extraction Function

This is the core of Day 4. One call does: ingest → function call → Pydantic validation.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = (
    "You are a senior tax compliance analyst at a Big4 firm.\n"
    "Extract structured tax data from the document provided by the user.\n\n"
    "CRITICAL RULES:\n"
    "1. Never hallucinate values. If a field is absent or unclear, omit it (set to null).\n"
    "2. Add the field name to missing_fields if you could not extract it reliably.\n"
    "3. Set extraction_confidence LOW (< 0.70) for approximate, ambiguous, or contradictory documents.\n"
    "4. Use extraction_notes to explain any assumptions you made.\n"
    "5. Amounts stated as approximate (\"roughly\", \"around\", \"maybe\") should be set to null — note in missing_fields."
)


def extract_tax_document(document_text, label=""):
    """
    Extract structured fields from an unstructured tax document using Azure function calling.
    Returns (TaxDocument | None, error_str | None).
    """
    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Extract from this document:\n\n{document_text}"},
        ],
        tools        = [EXTRACTION_FUNCTION_SCHEMA],
        tool_choice  = {
            "type": "function",
            "function": {"name": "extract_tax_document"}
        },
        temperature  = 0.0,
        max_tokens   = 800,
    )

    # Extract the function call arguments
    tool_calls = response.choices[0].message.tool_calls
    if not tool_calls:
        return None, "No function call returned by model"

    raw_args = tool_calls[0].function.arguments

    try:
        data = json.loads(raw_args)
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"

    try:
        result = TaxDocument.model_validate(data)
    except ValidationError as e:
        return None, f"Pydantic validation error: {e.errors()[0]['msg']}"

    if label:
        flag = " ⚠️  REVIEW" if result.human_review_required else "  ✅"
        print(f"{flag}  {label}")
        print(f"     entity:      {result.entity_name or 'MISSING'}")
        print(f"     period:      {result.tax_period or 'MISSING'}")
        print(f"     type:        {result.transaction_type or 'MISSING'}")
        if result.liability_amount:
            print(f"     amount:      {result.currency} {result.liability_amount:,.0f}")
        else:
            print(f"     amount:      MISSING")
        print(f"     confidence:  {result.extraction_confidence:.2f}")
        if result.missing_fields:
            print(f"     missing:     {result.missing_fields}")
        print()

    return result, None


print("✅ extract_tax_document() ready")

---
## 📄 Step 5: The 5 Test Documents

In [ ]:
TEST_DOCUMENTS = [
    {
        "id": 1,
        "label": "Clean email — well-structured",
        "text": """
From: priya.nair@techsolutions.in
Subject: GST filing — Q3 FY2025

Dear Tax Team,

TechSolutions Pvt Ltd has completed services worth Rs 8,50,000 for a registered
client in Bengaluru during Q3 FY2025. Both parties are GST-registered in Karnataka.
This is a B2B intra-state transaction.

Please confirm the applicable GST rate and filing timeline.

Regards, Priya
"""
    },
    {
        "id": 2,
        "label": "Ambiguous letter — approximate figures",
        "text": """
From: rajesh.kumar@alphacorp.in
Subject: GST query - urgent

Hi,

We need clarity on the GST for last quarter. Alpha Corp Pvt Ltd has done some
consulting work for a client in Dubai — roughly around 12 lakhs or so (haven't
finalised the invoice yet). Also some local work in Karnataka, maybe 8 lakhs?
Tax period should be Q3 FY25 but could be Q2 also, not 100% sure.

Please advise on what we need to file.

Thanks, Rajesh
"""
    },
    {
        "id": 3,
        "label": "Invoice snippet — missing period",
        "text": """
INVOICE
Vendor:   GlobalServ India Pvt Ltd, Mumbai
Client:   MNC Holdings LLC, New York, USA
Service:  IT infrastructure consulting
Amount:   USD 15,000
IGST:     0% (LUT filed - Ref: LUT/2024-25/0042)
Due date: 30 days from invoice date

Note: Payment to be received in USD to HDFC Bank A/c
"""
    },
    {
        "id": 4,
        "label": "Multi-entity document — two transactions",
        "text": """
Internal Tax Memo — October 2024

Two matters require attention this month:

1. Infovista Systems Ltd (Hyderabad) has raised an invoice of Rs 12,00,000 to
   DataCore Pvt Ltd (Chennai) for software implementation. Both are registered.
   This is an inter-state B2B supply. Period: Q2 FY2025.

2. Infovista also received legal advisory services from advocate K.P. Sharma
   (individual, unregistered) for Rs 75,000 during October 2024. RCM applies.

Both need to be reflected in the October GSTR-3B.
"""
    },
    {
        "id": 5,
        "label": "Near-empty message — extreme partial",
        "text": """
Hi,

Can you check if there are any GST issues with the recent payments?
Let me know.

- Amit
"""
    },
]

print(f"✅ {len(TEST_DOCUMENTS)} test documents loaded")
for d in TEST_DOCUMENTS:
    print(f"   Doc {d['id']}: {d['label']}")

In [ ]:
My_Document = [
    {
        "id": 1,
        "label": "Clean email — well-structured",
        "text": """
From: priya.nair@techsolutions.in
Subject: GST filing — Q3 FY2025

Dear Tax Team,

TechSolutions Pvt Ltd has completed services worth Rs 8,50,000 for a registered
client in Bengaluru during Q3 FY2025. Both parties are GST-registered in Karnataka.
This is a B2B intra-state transaction.

Please confirm the applicable GST rate and filing timeline.

Regards, Priya
"""
    },
]

print(f"✅ {len(My_Document)} test documents loaded")
for d in My_Document:
    print(f"   Doc {d['id']}: {d['label']}")

---
## 🚀 Step 6: Run the Extraction Engine on All 5 Documents

In [ ]:
print("EXTRACTION RESULTS")
print("=" * 65)

results      = []
parse_errors = []

for doc in TEST_DOCUMENTS:
    result, error = extract_tax_document(
        doc["text"],
        label=f"Doc {doc['id']}: {doc['label']}"
    )
    results.append(result)
    if error:
        parse_errors.append((doc["id"], error))
        print(f"  ❌ Doc {doc['id']}: {error}\n")

print("─" * 65)
print(f"Successful extractions : {len(results) - len(parse_errors)}/{len(TEST_DOCUMENTS)}")
print(f"Parse/validation errors: {len(parse_errors)}")
flagged = sum(1 for r in results if r and r.human_review_required)
print(f"Flagged for review     : {flagged}/{len(TEST_DOCUMENTS)}")

In [ ]:
print("EXTRACTION RESULTS - LIVE")
print("=" * 65)

results      = []
parse_errors = []

for doc in My_Document:
    result, error = extract_tax_document(
        doc["text"],
        label=f"Doc {doc['id']}: {doc['label']}"
    )
    results.append(result)
    if error:
        parse_errors.append((doc["id"], error))
        print(f"  ❌ Doc {doc['id']}: {error}\n")

print("─" * 65)
print(f"Successful extractions : {len(results) - len(parse_errors)}/{len(My_Document)}")
print(f"Parse/validation errors: {len(parse_errors)}")
flagged = sum(1 for r in results if r and r.human_review_required)
print(f"Flagged for review     : {flagged}/{len(My_Document)}")

---
## 🔍 Step 7: Deep Inspect Any Document

In [ ]:
# Change DOC_IDX (0–4) to inspect any document
DOC_IDX = 1   # default: the ambiguous Rajesh email

r = results[DOC_IDX]
if r is not None:
    print(f"DOCUMENT {DOC_IDX+1}: {TEST_DOCUMENTS[DOC_IDX]['label']}")
    print("-" * 60)
    print("Original text:")
    print(TEST_DOCUMENTS[DOC_IDX]["text"])
    print("\nExtracted fields:")
    for field, value in r.model_dump().items():
        marker = ""
        if field == "missing_fields" and value:
            marker = " ← TO CHASE"
        if field == "human_review_required" and value:
            marker = " ← AUTO-FLAGGED"
        print(f"  {field:<28} : {value}{marker}")
else:
    print(f"Document {DOC_IDX+1} failed to parse.")

---
## 📊 Step 8: Extraction Quality Summary

In [ ]:
print("EXTRACTION QUALITY SUMMARY")
print(f"{'Doc':<6} {'Label':<38} {'Conf':>6} {'Missing':>8} {'Review':>8}")
print("-" * 70)

for i, (doc, result) in enumerate(zip(TEST_DOCUMENTS, results)):
    if result:
        bar    = "█" * int(result.extraction_confidence * 10)
        n_miss = len(result.missing_fields)
        review = "YES ⚠️" if result.human_review_required else "no"
        print(f"  {doc['id']:<4} {doc['label']:<38} {result.extraction_confidence:.2f}  {n_miss:>6}   {review}")
    else:
        print(f"  {doc['id']:<4} {doc['label']:<38} PARSE ERROR")

valid = [r for r in results if r]
if valid:
    avg = sum(r.extraction_confidence for r in valid) / len(valid)
    print(f"\nAverage extraction confidence: {avg:.2f}")
    print(f"Docs needing human review    : {sum(1 for r in valid if r.human_review_required)}/{len(valid)}")

---
## 💾 Step 9: Save Raw Results

In [ ]:
output = {
    "notebook":    "Day4B_Extraction_Engine",
    "exported_at": datetime.datetime.now().isoformat(),
    "model":       AZURE_DEPLOYMENT_NAME,
    "results": [
        {
            "doc_id":  TEST_DOCUMENTS[i]["id"],
            "label":   TEST_DOCUMENTS[i]["label"],
            "extraction": results[i].model_dump() if results[i] else None,
            "error":  None if results[i] else "parse_failed",
        }
        for i in range(len(TEST_DOCUMENTS))
    ]
}

with open("day4b_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)

print("✅ Results saved to day4b_results.json")

---
## 🎓 What This Notebook Demonstrated

| Capability | Notes |
|---|---|
| Function calling enforces schema | No fences, no deviation, no parsing dance |
| Partial extraction on ambiguous docs | `missing_fields` tracks what to chase |
| Confidence auto-gates human review | Low-confidence records flagged before DB insert |
| Multi-entity doc (Doc 4) | Model extracts primary entity — notes secondary in `extraction_notes` |
| Near-empty doc (Doc 5) | Graceful degradation — doesn't crash, returns high `missing_fields` |

---
## ➡️ Next: Notebook 4C — Validation Pipeline

Wire error handling (retry on parse fail, partial return on validation fail),
route high-confidence records to a mock DB insert, and export the complete audit trail.